# Credit Risk | Exploratory Data Analysis

**Goal:** explore a synthetic credit portfolio and prepare next-month individual default labels. Runs with CSV files or PostgreSQL.


## 1. Load data

Required packages: `pandas`, `matplotlib`, `sqlalchemy`, `psycopg`, and `python-dotenv`.

In [ ]:
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from dotenv import load_dotenv
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

# Project paths: notebooks work when Jupyter starts from the repository root or notebooks/.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

load_dotenv(PROJECT_ROOT / ".env")

plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True, "grid.alpha": 0.2})


In [ ]:
# Use PostgreSQL when credentials are available; otherwise load the provided synthetic CSVs.
required = ["DB_USER", "DB_PASSWORD", "DB_HOST", "DB_NAME"]
if all(os.getenv(key) for key in required):
    db_url = URL.create(
        "postgresql+psycopg",
        username=os.getenv("DB_USER"), password=os.getenv("DB_PASSWORD"),
        host=os.getenv("DB_HOST"), port=int(os.getenv("DB_PORT", "5432")),
        database=os.getenv("DB_NAME"),
    )
    engine = create_engine(db_url, pool_pre_ping=True)
    contracts = pd.read_sql("SELECT * FROM raw.contracts", engine)
    performance = pd.read_sql("SELECT * FROM raw.monthly_performance", engine)
    print("Source: PostgreSQL")
else:
    contracts = pd.read_csv(PROJECT_ROOT / "data/raw/contratos.csv")
    performance = pd.read_csv(PROJECT_ROOT / "data/raw/desempenho_mensal.csv")
    contracts = contracts.rename(columns={
        "segmento": "segment", "data_contratacao": "origination_date",
        "valor_original": "original_value", "taxa_juros_anual": "annual_int_rate",
        "prazo_meses": "months_term", "lgd_estimada": "lgd_est",
    })
    performance = performance.rename(columns={
        "mes_referencia": "reference_month", "saldo_devedor": "outstanding_balance",
        "dias_atraso": "days_past_due", "novo_default": "new_default",
    })
    print("Source: bundled synthetic CSVs")

contracts["origination_date"] = pd.to_datetime(contracts["origination_date"])
performance["reference_month"] = pd.to_datetime(performance["reference_month"])
performance = performance.sort_values(["loan_id", "reference_month"])

print(f"Loans: {len(contracts):,}")
print(f"Loan-month records: {len(performance):,}")
print(f"History: {performance.reference_month.min():%b %Y} – {performance.reference_month.max():%b %Y}")
display(contracts.head(), performance.head())


## 2. Data quality

Check missing values, duplicates, and the most important business rules before analyzing defaults.

In [ ]:
print("Missing values — contracts:")
display(contracts.isna().sum().to_frame("missing"))

print("Missing values — monthly performance:")
display(performance.isna().sum().to_frame("missing"))

checks = pd.Series({
    "Duplicate loan IDs": contracts.duplicated("loan_id").sum(),
    "Duplicate loan-months": performance.duplicated(["loan_id", "reference_month"]).sum(),
    "Unknown loan IDs": (~performance.loan_id.isin(contracts.loan_id)).sum(),
    "Negative balances": (performance.outstanding_balance < 0).sum(),
    "Invalid LGD values": (~contracts.lgd_est.between(0, 1)).sum(),
    "Invalid default labels": (~performance.new_default.isin([0, 1])).sum(),
    "Loans with multiple new defaults": (performance.groupby("loan_id").new_default.sum() > 1).sum(),
})
display(checks.to_frame("issues"))

In [ ]:
display(
    contracts[["original_value", "annual_int_rate", "months_term", "lgd_est"]].describe().round(2),
    performance[["outstanding_balance", "days_past_due", "new_default"]].describe().round(2)
)

## 3. Portfolio overview

How many loans belong to each segment and rating? How much was originally lent?

In [ ]:
portfolio = contracts.groupby("segment").agg(
    loans=("loan_id", "count"),
    original_amount=("original_value", "sum"),
    average_interest=("annual_int_rate", "mean"),
)
portfolio["average_interest"] *= 100

display(portfolio.round(2))

contracts["segment"].value_counts().plot.bar(title="Loans by segment", rot=0)
plt.ylabel("Number of loans")
plt.show()

In [ ]:
contracts["rating"].value_counts().reindex(list("ABCDE"), fill_value=0).plot.bar(
    title="Loans by credit rating", rot=0
)
plt.xlabel("Rating")
plt.ylabel("Number of loans")
plt.show()

contracts["original_value"].plot.hist(bins=35, title="Original loan amount")
plt.xlabel("BRL")
plt.show()

## 4. Default trends

The **monthly new-default rate** is the number of new defaults divided by the loans observed that month. This is not the same as an annual probability of default.

In [ ]:
monthly_risk = performance.groupby("reference_month").agg(
    active_loans=("loan_id", "count"),
    new_defaults=("new_default", "sum"),
    outstanding_balance=("outstanding_balance", "sum"),
)
monthly_risk["default_rate_pct"] = (
    100 * monthly_risk["new_defaults"] / monthly_risk["active_loans"]
)

display(monthly_risk.tail(6).round(2))

In [ ]:
monthly_risk[["active_loans", "new_defaults"]].plot(
    subplots=True, figsize=(10, 6), title=["Loans observed each month", "New defaults per month"]
)
plt.tight_layout()
plt.show()

monthly_risk["default_rate_pct"].plot(title="Monthly new-default rate (%)")
plt.ylabel("Rate (%)")
plt.show()

## 5. Risk by segment, rating, and payment status

Join the tables to explore where defaults are concentrated. For these summaries, rates count **loan-month observations**, not unique loans.

In [ ]:
loan_month = performance.merge(contracts, on="loan_id", validate="many_to_one")

segment_risk = loan_month.groupby("segment").agg(
    observations=("loan_id", "count"),
    new_defaults=("new_default", "sum"),
)
segment_risk["rate_pct"] = 100 * segment_risk.new_defaults / segment_risk.observations

rating_risk = loan_month.groupby("rating").agg(
    observations=("loan_id", "count"),
    new_defaults=("new_default", "sum"),
).reindex(list("ABCDE"))
rating_risk["rate_pct"] = 100 * rating_risk.new_defaults / rating_risk.observations

display(segment_risk.round(2), rating_risk.round(2))

In [ ]:
segment_risk["rate_pct"].plot.bar(title="Monthly default incidence by segment", rot=0)
plt.ylabel("New defaults (%)")
plt.show()

rating_risk["rate_pct"].plot.bar(title="Monthly default incidence by rating", rot=0)
plt.ylabel("New defaults (%)")
plt.show()

In [ ]:
dpd = performance.groupby("days_past_due").agg(
    observations=("loan_id", "count"),
    new_defaults=("new_default", "sum"),
)
display(dpd)

dpd["observations"].plot.bar(title="Days past due (DPD)", rot=0)
plt.xlabel("Days past due")
plt.ylabel("Loan-month records")
plt.show()

## 6. Prepare the target for individual PD

**Prediction question:** at the end of month **t**, what is the probability that this loan will enter default in **t + 1**?

Use information available at **t** as features and `new_default` from **t + 1** as the label. Keep only consecutive observed months, and exclude loans that have already defaulted at **t**.

> Caution: loans without a recorded next month are *unlabeled*, not automatically non-defaults. This sample may have selection bias.

In [ ]:
grouped = performance.groupby("loan_id")

next_month = grouped["reference_month"].shift(-1)
next_default = grouped["new_default"].shift(-1)

has_next_month = next_month.eq(
    performance["reference_month"] + pd.DateOffset(months=1)
)
eligible = performance["new_default"].eq(0) & has_next_month

pd_dataset = performance.loc[eligible].copy()
pd_dataset["default_next_month"] = next_default.loc[eligible].astype(int)
pd_dataset = pd_dataset.merge(contracts, on="loan_id", validate="many_to_one")

pd_dataset["months_on_book"] = (
    (pd_dataset.reference_month.dt.year - pd_dataset.origination_date.dt.year) * 12
    + pd_dataset.reference_month.dt.month - pd_dataset.origination_date.dt.month
)

print(f"Labeled observations: {len(pd_dataset):,}")
print(f"Defaults next month: {pd_dataset.default_next_month.sum():,}")
print(f"Positive class: {pd_dataset.default_next_month.mean():.2%}")
display(pd_dataset.head())

In [ ]:
# Which features appear related to default in the following month?
for column in ["rating", "segment", "days_past_due"]:
    summary = pd_dataset.groupby(column)["default_next_month"].agg(
        observations="count", defaults="sum", rate="mean"
    )
    summary["rate_pct"] = summary["rate"] * 100
    print(f"Next-month default by {column}:")
    display(summary.round(2))

In [ ]:
pd_dataset.groupby("rating")["default_next_month"].mean().reindex(list("ABCDE")).mul(100).plot.bar(
    title="Next-month default rate by rating", rot=0
)
plt.ylabel("Next-month default (%)")
plt.show()

pd_dataset.groupby("days_past_due")["default_next_month"].mean().mul(100).plot.bar(
    title="Next-month default rate by days past due", rot=0
)
plt.ylabel("Next-month default (%)")
plt.show()

## 7. Ready for Machine Learning

This is an **imbalanced classification** problem. We will use a time-based train/test split and compare Logistic Regression, Random Forest, and XGBoost using **PR-AUC, ROC-AUC, and probability calibration**.

`loan_id` and dates help identify samples but should not be fed directly as numeric features. Never use next month's DPD or default as an input feature.

In [ ]:
feature_cols = [
    "segment", "rating", "original_value", "annual_int_rate",
    "months_term", "lgd_est", "outstanding_balance",
    "days_past_due", "months_on_book"
]
target_col = "default_next_month"

# Hold out the last six labeled months for evaluation.
test_start = sorted(pd_dataset.reference_month.unique())[-6]
train_pd = pd_dataset[pd_dataset.reference_month < test_start].copy()
test_pd = pd_dataset[pd_dataset.reference_month >= test_start].copy()

split_summary = pd.DataFrame({
    "sample": ["Train", "Test"],
    "rows": [len(train_pd), len(test_pd)],
    "next_month_defaults": [train_pd[target_col].sum(), test_pd[target_col].sum()],
    "default_rate_pct": [train_pd[target_col].mean() * 100, test_pd[target_col].mean() * 100],
})
display(split_summary.round(2))

print("Features:", feature_cols)
print("Target:", target_col)

### Takeaways

- **Portfolio:** segments, ratings, principal, and payment status are potential risk drivers.
- **Outcome:** `default_next_month` labels actual *new* defaults one month ahead.
- **Evaluation:** keep future months out of training; accuracy alone is misleading for rare defaults.
- **Limitations:** the data are synthetic and default is simplified to the recorded event; deployable PD models need stronger validation.

**Next notebook:** train and compare individual-loan PD classifiers. **Forecasting of the aggregate default rate is a separate optional exercise.**

In [ ]:
output_path = PROJECT_ROOT / "data/processed/pd_dataset.csv"
pd_dataset.to_csv(output_path, index=False)
print(f"Saved {len(pd_dataset):,} labeled observations to {output_path.relative_to(PROJECT_ROOT)}")
